In [2]:
import os
from pathlib import Path

# Suche im Hugging Face Cache nach .pt oder .safetensors Dateien
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
checkpoints = list(cache_dir.glob("**/sam3*.pt")) + list(cache_dir.glob("**/sam3*.safetensors"))

if checkpoints:
    print("Gefundene Modelle im Cache:")
    for c in checkpoints:
        print(c)
else:
    print("Kein Modell im Cache gefunden.")

Gefundene Modelle im Cache:
/home/xxbananaopxx/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt
/home/xxbananaopxx/.cache/huggingface/hub/models--facebook--sam3.1/snapshots/daa63191845a41281374e725f4c9e51c7a824460/sam3.1_multiplex.pt
/home/xxbananaopxx/.cache/huggingface/hub/models--facebook--sam3.1/.no_exist/daa63191845a41281374e725f4c9e51c7a824460/sam3.pt


In [5]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
import torch

# 1. DEN PFAD ZUR GEFUNDENEN DATEI EINTRAGEN
# Beispiel: "/mnt/c/Users/sinke/Desktop/Masterarbeit/sam3/sam3/checkpoints/sam3_hiera_tiny.pt"
MEIN_MODELL_PFAD = "/home/xxbananaopxx/.cache/huggingface/hub/models--facebook--sam3/snapshots/3c879f39826c281e95690f02c7821c4de09afae7/sam3.pt" 
MEIN_VIDEO_PFAD = "./videos/fishvideo1.mp4" # Pfad zu deinem Video

# 2. Predictor initialisieren mit dem richtigen Modell-Pfad
# Wir setzen imgsz=1024 (Standard für SAM) oder 640 für mehr Speed
overrides = dict(
    conf=0.25, 
    task="segment", 
    mode="predict", 
    imgsz=1024, 
    model=MEIN_MODELL_PFAD, # Hier wird die Datei jetzt geladen
    half=True,               # Nutzt FP16 für weniger VRAM (gut für deine RTX 2070)
    device="cuda"            # Explizit die GPU nutzen
)
predictor = SAM3VideoSemanticPredictor(overrides=overrides)

# 3. Tracking starten
# 'text' ist der Prompt für deine Fische
results = predictor(source=MEIN_VIDEO_PFAD, text=["fish"], stream=True)

# 4. Ergebnisse verarbeiten
for r in results:
    # r.show() öffnet ein Fenster (funktioniert oft nicht gut in WSL/Notebooks)
    # r.plot() erzeugt ein Bild mit Masken, das du anzeigen oder speichern kannst
    #im_bgr = r.plot() 
    r.show()
    
    # In einem Jupyter Notebook kannst du das Bild so anzeigen:
    # from PIL import Image
    # display(Image.fromarray(im_bgr[..., ::-1])) 
    
    print(f"Frame verarbeitet - Objekte gefunden: {len(r.masks) if r.masks else 0}")


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2070 SUPER, 8192MiB)
WARNING ⚠️ imgsz=[1024] must be multiple of max stride 14, updating to [1036]
video 1/1 (frame 1/318) /mnt/c/Users/sinke/Desktop/Masterarbeit/sam3/videos/fishvideo1.mp4: 1036x1036 25 fishs, 1201.7ms
Frame verarbeitet - Objekte gefunden: 25


No applications found for mimetype: image/png
.

video 1/1 (frame 2/318) /mnt/c/Users/sinke/Desktop/Masterarbeit/sam3/videos/fishvideo1.mp4: 1036x1036 29 fishs, 2148.8ms
Frame verarbeitet - Objekte gefunden: 29


No applications found for mimetype: image/png
.

video 1/1 (frame 3/318) /mnt/c/Users/sinke/Desktop/Masterarbeit/sam3/videos/fishvideo1.mp4: 1036x1036 35 fishs, 3717.8ms
Frame verarbeitet - Objekte gefunden: 35


No applications found for mimetype: image/png
.

KeyboardInterrupt: 

In [6]:
# Listet alle öffentlichen Methoden und Attribute auf
methods = [method for method in dir(predictor) if not method.startswith('_')]
print(methods)

['ALWAYS_OCCLUDED', 'CONFIRMED', 'HIGH_CONF_THRESH', 'HIGH_IOU_THRESH', 'NEVER_OCCLUDED', 'NO_OBJ_LOGIT', 'UNCONFIRMED', 'add_callback', 'add_prompt', 'args', 'assoc_iou_thresh', 'batch', 'build_outputs', 'callbacks', 'data', 'dataset', 'decrease_trk_keep_alive_for_empty_masklets', 'det_nms_thresh', 'device', 'done_warmup', 'features', 'fill_hole_area', 'generate', 'get_im_features', 'get_model', 'hotstart_delay', 'hotstart_dup_thresh', 'hotstart_unmatch_thresh', 'im', 'imgsz', 'inference', 'inference_features', 'inference_state', 'init_state', 'init_trk_keep_alive', 'interpol_size', 'masklet_confirmation_consecutive_det_thresh', 'masklet_confirmation_enable', 'max_num_objects', 'max_trk_keep_alive', 'mean', 'min_trk_keep_alive', 'model', 'new_det_thresh', 'num_obj_for_compile', 'o2o_matching_masklets_enable', 'plotted_img', 'postprocess', 'pre_transform', 'predict_cli', 'preprocess', 'prompt_inference', 'prompts', 'recondition_every_nth_frame', 'reconstruction_bbox_det_score', 'recons

In [7]:
help(predictor.add_prompt)

Help on method add_prompt in module ultralytics.models.sam.predict:

add_prompt(frame_idx, text=None, bboxes=None, labels=None, inference_state=None) method of ultralytics.models.sam.predict.SAM3VideoSemanticPredictor instance
    Add text, point or box prompts on a single frame. This method returns the inference outputs only on the
    prompted frame.

    Note that text prompts are NOT associated with a particular frame (i.e. they apply
    to all frames). However, we only run inference on the frame specified in `frame_idx`.

